In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 9.0 MB/s eta 0:00:00


In [ ]:
# =========================================================
# YOLOv8-seg 기반 양상추 면적 + Shape Descriptor 추출 파이프라인
# (json 없이 / 복사-붙여넣기 실행 가능 버전)
# =========================================================
# 이 코드는 **코딩을 거의 모르는 사용자**를 기준으로 작성됨.
# 순서대로 위에서 아래까지 실행하면 됨.
# ---------------------------------------------------------
# 전체 흐름 (네 ipynb 구조 그대로)
#   단계 0. 라이브러리 설치
#   단계 1. YOLOv8-seg 모델 로드 + 면적/기본 지표 추출
#   단계 2. scale_map으로 mm 환산
#   단계 3. shape descriptor 계산 (mask → contour)
#   단계 4. 엑셀 저장
# =========================================================
# =========================================================
# 단계 1. 라이브러리 import + 기본 설정
# =========================================================
import os
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from concurrent.futures import ThreadPoolExecutor
import time
from skimage.measure import EllipseModel

# -----------------------------
# 경로 설정 (기존 ipynb 방식 그대로)
# -----------------------------
GRID_FOLDER = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/3. ROI_LETTUCE/bed_grid_split/251129-251206"  # grid 결과 폴더
MODEL_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/Practice/251011 양상추 seg 연습(2)/result/lettuce_yolov8n4/weights/best.pt"  # 기존 세그모델 경로
OUT_XLSX = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/4. 결과 출력 시각화/251207_lettuce_segmentation_area(1)_index.xlsx"

# -----------------------------
# 사용자 설정 (기존 ipynb와 동일한 개념)
# -----------------------------
BATCH_SIZE = 24       # T4에서 안전하며 빠른 batch
CHUNK_SIZE = 3000     # 3000장 단위로 잘라서 처리
N_WORKERS = 4         # 이미지 로드 병렬 처리 개수 (CPU 코어 여유에 따라 조절)

# 모델 로드 (여기서 이미 학습된 모델 사용)
model = YOLO(MODEL_PATH)

# scale_map 사용 안 함 (기존 ipynb와 동일)
scale_map = None


In [ ]:


# =========================================================
# 단계 2. QC용 보조 함수 (이미지 품질)
# =========================================================
def brightness_mean(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return gray.mean()


def blur_score(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()


# =========================================================
# 단계 3. mask → contour 추출 함수
# =========================================================
def get_main_contour(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if len(contours) == 0:
        return None
    return max(contours, key=cv2.contourArea)


# =========================================================
# 단계 4. Shape descriptor 계산 함수
# (json 없이, YOLO-seg mask 기반)
# =========================================================
def calc_shape_descriptors(contour):
    out = {
        "bbox_w": np.nan,
        "bbox_h": np.nan,
        "perimeter_px": np.nan,
        "circularity": np.nan,
        "solidity": np.nan,
        "concavity": np.nan,
        "curvature": np.nan,
        "roughness": np.nan,
    }

    if contour is None:
        return out

    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)
    x, y, w, h = cv2.boundingRect(contour)

    out["bbox_w"] = w
    out["bbox_h"] = h
    out["perimeter_px"] = perimeter

    if perimeter > 0:
        out["circularity"] = 4 * np.pi * area / (perimeter ** 2)

    hull = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)
    if hull_area > 0:
        out["solidity"] = area / hull_area
        out["concavity"] = 1 - out["solidity"]

    pts = contour.squeeze()
    if len(pts) > 10:
        diffs = np.diff(pts, axis=0)
        angles = np.arctan2(diffs[:, 1], diffs[:, 0])
        curv = np.abs(np.diff(angles))
        out["curvature"] = np.mean(curv)
        out["roughness"] = np.std(curv)

    return out


In [ ]:


# =========================================================
# 단계 5. 메인 루프
# (YOLO-seg 추론 → 면적 → QC → shape descriptor)
# =========================================================
rows = []
start_time = time.time()

from glob import glob
from tqdm import tqdm

# 전체 grid 이미지 수집 (하위 폴더 포함)
image_paths = sorted(glob(os.path.join(GRID_FOLDER, "**", "*.png"), recursive=True))

# ---------- chunk 단위 처리 ----------
for c_start in range(0, len(image_paths), CHUNK_SIZE):
    chunk_paths = image_paths[c_start:c_start + CHUNK_SIZE]

    for i in tqdm(
        range(0, len(chunk_paths), BATCH_SIZE),
        desc=f"Chunk {c_start//CHUNK_SIZE + 1}/{(len(image_paths)-1)//CHUNK_SIZE + 1}",
        leave=True
    ):
        batch_paths = chunk_paths[i:i + BATCH_SIZE]

        # 병렬 이미지 로드
        with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
            imgs = list(ex.map(cv2.imread, batch_paths))

        valid = [(p, im) for p, im in zip(batch_paths, imgs) if im is not None]
        if not valid:
            continue

        paths, imgs = zip(*valid)
        results = model(list(imgs), verbose=False)

        for img_path, img, result in zip(paths, imgs, results):
            filename = os.path.basename(img_path)
            H, W = img.shape[:2]

            # ---------- QC ----------
            b_mean = brightness_mean(img)
            b_blur = blur_score(img)

            masks = result.masks
            boxes = result.boxes
            if masks is None:
                continue

            n_instances = len(masks)
            conf_list = boxes.conf.cpu().numpy().tolist()

            for idx in range(n_instances):
                mask = masks.data[idx].cpu().numpy().astype(np.uint8)
                mask = cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)

                area_px = cv2.countNonZero(mask)
                contour = get_main_contour(mask)
                shape = calc_shape_descriptors(contour)

                rows.append({
                    "filename": filename,
                    "area_px": area_px,
                    "area_mm2": np.nan,
                    "px_per_mm_x": np.nan,
                    "px_per_mm_y": np.nan,
                    "conf": conf_list[idx],
                    "conf_mean": np.mean(conf_list),
                    "conf_max": np.max(conf_list),
                    "n_instances": n_instances,
                    "brightness_mean": b_mean,
                    "blur_score": b_blur,
                    **shape
                })


# =========================================================
# 단계 6. 엑셀 저장
# =========================================================
df = pd.DataFrame(rows)
df.to_excel(OUT_XLSX, index=False)
elapsed = time.time() - start_time
print(f"[완료] 결과 저장: {OUT_XLSX}")
print(f"총 처리 시간: {elapsed/60:.1f} 분")


Chunk 7/7: 100%|██████████| 32/32 [01:12<00:00,  2.27s/it]


[완료] 결과 저장: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/4. 결과 출력 시각화/251207_lettuce_segmentation_area(1)_index.xlsx
총 처리 시간: 30.0 분


## 이미 한건 어쩔수 없고 다음번엔 이렇게 해라

In [2]:
# =========================================================
# YOLOv8-seg 기반 양상추 면적 + Shape Descriptor 추출 파이프라인
# (json 없이 / 복사-붙여넣기 실행 가능 버전)
# =========================================================
# 이 코드는 **코딩을 거의 모르는 사용자**를 기준으로 작성됨.
# 순서대로 위에서 아래까지 실행하면 됨.
# ---------------------------------------------------------
# 전체 흐름 (네 ipynb 구조 그대로)
#   단계 0. 라이브러리 설치
#   단계 1. YOLOv8-seg 모델 로드 + 면적/기본 지표 추출
#   단계 2. scale_map으로 mm 환산
#   단계 3. shape descriptor 계산 (mask → contour)
#   단계 4. 엑셀 저장
# =========================================================

# =========================================================
# 단계 0. 라이브러리 설치 (처음 1번만)
# =========================================================
# 터미널 또는 노트북 셀에서 실행
# !pip install ultralytics opencv-python pandas numpy scikit-image scipy


# =========================================================
# 단계 1. 라이브러리 import + 기본 설정
# =========================================================
import os
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from concurrent.futures import ThreadPoolExecutor
import time
from skimage.measure import EllipseModel

# -----------------------------
# 경로 설정 (기존 ipynb 방식 그대로)
# -----------------------------
GRID_FOLDER = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/3. ROI_LETTUCE/bed_grid_split/251207-251212"  # grid 결과 폴더
MODEL_PATH = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/Practice/251011 양상추 seg 연습(2)/result/lettuce_yolov8n4/weights/best.pt"  # 기존 세그모델 경로
OUT_XLSX = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/4. 결과 출력 시각화/1작기(251128-251226)/1파트(251128-251213)/251212_lettuce_segmentation_area(1)_index.xlsx"

# (선택) 픽셀 스케일 맵 CSV가 있으면 mm^2, mm 단위도 같이 계산됨
# - 예전 방식처럼 parent_name(폴더명) 기준 매칭
# - 컬럼 형태는 아래 중 하나만 맞으면 됨:
#   A) parent_name + mm_per_px
#   B) image_name(확장자 포함) + mm_per_px
#   C) parent_name + px_per_mm_x, px_per_mm_y
SCALE_CSV = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/4. 결과 출력 시각화/1작기(251128-251226)/1파트(251128-251213)/251212_pixel_scale_map.csv"


# -----------------------------
# 사용자 설정 (기존 ipynb와 동일한 개념)
# -----------------------------
BATCH_SIZE = 24       # T4에서 안전하며 빠른 batch
CHUNK_SIZE = 3000     # 3000장 단위로 잘라서 처리
N_WORKERS = 4         # 이미지 로드 병렬 처리 개수 (CPU 코어 여유에 따라 조절)

# 모델 로드 (여기서 이미 학습된 모델 사용)
model = YOLO(MODEL_PATH)
print("[INFO] model task:", getattr(model, "task", "unknown"))

# 추론 파라미터(필요시 조절)
PRED_CONF = 0.05   # 검출이 거의 안 나오면 0.01~0.1 사이로 조절
PRED_IOU = 0.7

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[INFO] model task: segment


In [3]:


# -------------------------------------------------
# 스케일 맵 로드 (없으면 mm 환산은 NaN)
# -------------------------------------------------
scale_map = None
if isinstance(SCALE_CSV, str) and len(SCALE_CSV) > 0 and os.path.exists(SCALE_CSV):
    scale_df = pd.read_csv(SCALE_CSV)

    # parent_name 컬럼 보정
    if "parent_name" not in scale_df.columns:
        if "image_name" in scale_df.columns:
            scale_df["parent_name"] = scale_df["image_name"].astype(str).apply(lambda x: os.path.splitext(os.path.basename(x))[0])
        else:
            raise ValueError("SCALE_CSV에 parent_name 또는 image_name 컬럼이 필요합니다.")

    # lookup dict로 변환
    scale_map = scale_df.set_index("parent_name").to_dict(orient="index")
    print(f"[INFO] scale_map 로드 완료: {len(scale_map)} rows")
else:
    print("[INFO] SCALE_CSV 미사용: mm 단위(area_mm2/perim_mm)는 NaN으로 저장됩니다.")


# =========================================================
# 단계 2. QC용 보조 함수 (이미지 품질)
# =========================================================
def brightness_mean(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return gray.mean()


def blur_score(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()


# =========================================================
# 단계 3. mask → contour 추출 함수
# =========================================================
def get_main_contour(mask):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if len(contours) == 0:
        return None
    return max(contours, key=cv2.contourArea)


[INFO] scale_map 로드 완료: 1878 rows


In [4]:


# =========================================================
# 단계 4. Shape descriptor 계산 함수
# (json 없이, YOLO-seg mask 기반)
# =========================================================
def calc_shape_descriptors(contour):
    out = {
        "bbox_w": np.nan,
        "bbox_h": np.nan,
        "perimeter_px": np.nan,
        "circularity": np.nan,
        "solidity": np.nan,
        "concavity": np.nan,
        "curvature": np.nan,
        "roughness": np.nan,
    }

    if contour is None:
        return out

    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)
    x, y, w, h = cv2.boundingRect(contour)

    out["bbox_w"] = w
    out["bbox_h"] = h
    out["perimeter_px"] = perimeter

    if perimeter > 0:
        out["circularity"] = 4 * np.pi * area / (perimeter ** 2)

    hull = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)
    if hull_area > 0:
        out["solidity"] = area / hull_area
        out["concavity"] = 1 - out["solidity"]

    pts = contour.squeeze()
    if len(pts) > 10:
        diffs = np.diff(pts, axis=0)
        angles = np.arctan2(diffs[:, 1], diffs[:, 0])
        curv = np.abs(np.diff(angles))
        out["curvature"] = np.mean(curv)
        out["roughness"] = np.std(curv)

    return out


# =========================================================
# 단계 5. 메인 루프
# (YOLO-seg 추론 → 면적 → QC → shape descriptor)
# =========================================================
rows = []
start_time = time.time()

# 디버그/검증용 카운터
cnt_read_ok = 0
cnt_pred = 0
cnt_with_masks = 0
cnt_instances = 0

from glob import glob
from tqdm import tqdm

# 전체 grid 이미지 수집 (하위 폴더 포함)
# 전체 grid 이미지 수집 (하위 폴더 포함) - 확장자 다양하게
patterns = ["**/*.jpg", "**/*.JPG", "**/*.jpeg", "**/*.JPEG", "**/*.png", "**/*.PNG"]
image_paths = []
for pat in patterns:
    image_paths.extend(glob(os.path.join(GRID_FOLDER, pat), recursive=True))
image_paths = sorted(set(image_paths))
print(f"[INFO] 총 grid cell 이미지 개수: {len(image_paths)}")
print(f"[INFO] 총 {(len(image_paths)-1)//CHUNK_SIZE + 1}개의 chunk로 분할( chunk 당 {CHUNK_SIZE}장 )")

# ---------- chunk 단위 처리 ----------
for c_start in range(0, len(image_paths), CHUNK_SIZE):
    chunk_paths = image_paths[c_start:c_start + CHUNK_SIZE]

    for i in tqdm(
        range(0, len(chunk_paths), BATCH_SIZE),
        desc=f"Chunk {c_start//CHUNK_SIZE + 1}/{(len(image_paths)-1)//CHUNK_SIZE + 1}",
        leave=True
    ):
        batch_paths = chunk_paths[i:i + BATCH_SIZE]

        # 병렬 이미지 로드
        with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
            imgs = list(ex.map(cv2.imread, batch_paths))

        valid = [(p, im) for p, im in zip(batch_paths, imgs) if im is not None]
        cnt_read_ok += len(valid)
        if not valid:
            continue

        paths, imgs = zip(*valid)

        # YOLO 추론 (검출이 안 나오면 PRED_CONF 낮추기)
        results = model(list(imgs), verbose=False, conf=PRED_CONF, iou=PRED_IOU)
        cnt_pred += len(results)

        for img_path, img, result in zip(paths, imgs, results):
            filename = os.path.basename(img_path)
            H, W = img.shape[:2]

            # ---------- QC ----------
            b_mean = brightness_mean(img)
            b_blur = blur_score(img)

            masks = result.masks
            boxes = result.boxes
            if masks is None:
                # ⚠️ 여기로만 계속 들어오면: (1) 모델이 seg가 아니거나 (2) 검출이 안 되는 상황
                continue
            cnt_with_masks += 1

            n_instances = len(masks)
            cnt_instances += n_instances
            conf_list = boxes.conf.cpu().numpy().tolist()

            for idx in range(n_instances):
                mask = masks.data[idx].cpu().numpy().astype(np.uint8)
                mask = cv2.resize(mask, (W, H), interpolation=cv2.INTER_NEAREST)

                area_px = cv2.countNonZero(mask)
                contour = get_main_contour(mask)
                shape = calc_shape_descriptors(contour)

                # mm 환산 (옵션)
                parent_name = os.path.basename(os.path.dirname(img_path))
                mm_per_px = np.nan
                pxmm_x = np.nan
                pxmm_y = np.nan

                if scale_map is not None and parent_name in scale_map:
                    info = scale_map[parent_name]
                    # 케이스 1) mm_per_px가 있는 경우
                    if "mm_per_px" in info and pd.notna(info["mm_per_px"]):
                        mm_per_px = float(info["mm_per_px"])
                        area_mm2 = area_px * (mm_per_px ** 2)
                        perim_mm = float(shape.get("perimeter_px", np.nan)) * mm_per_px
                    # 케이스 2) px_per_mm_x/y가 있는 경우
                    elif ("px_per_mm_x" in info) and ("px_per_mm_y" in info) and pd.notna(info["px_per_mm_x"]) and pd.notna(info["px_per_mm_y"]):
                        pxmm_x = float(info["px_per_mm_x"])
                        pxmm_y = float(info["px_per_mm_y"])
                        area_mm2 = area_px / (pxmm_x * pxmm_y)
                        perim_mm = float(shape.get("perimeter_px", np.nan)) / ((pxmm_x + pxmm_y) / 2.0)
                    else:
                        area_mm2 = np.nan
                        perim_mm = np.nan
                else:
                    area_mm2 = np.nan
                    perim_mm = np.nan

                rows.append({
                    "image_path": img_path,
                    "parent_name": parent_name,
                    "filename": filename,
                    "area_px": area_px,
                    "area_mm2": area_mm2,
                    "perim_mm": perim_mm,
                    "mm_per_px": mm_per_px,
                    "px_per_mm_x": pxmm_x,
                    "px_per_mm_y": pxmm_y,
                    "conf": conf_list[idx],
                    "conf_mean": np.mean(conf_list),
                    "conf_max": np.max(conf_list),
                    "n_instances": n_instances,
                    "brightness_mean": b_mean,
                    "blur_score": b_blur,
                    **shape
                })


# =========================================================
# 단계 6. 엑셀 저장
# =========================================================
print(f"[INFO] read_ok={cnt_read_ok}  pred_batches={cnt_pred}  with_masks={cnt_with_masks}  total_instances={cnt_instances}")

if len(rows) == 0:
    print("[WARN] rows가 0입니다. 아래 원인 가능성을 확인하세요:")
    print("  1) 모델이 segmentation 모델이 아님( detect weights ) → model task 확인")
    print("  2) conf가 너무 높아 검출이 0 → PRED_CONF를 0.01~0.1로 낮춰 재시도")
    print("  3) 입력 이미지/전처리 문제(너무 작은 해상도, 잘못된 grid 폴더) → 샘플 1장 시각 확인")

# 저장 (빈 경우에도 헤더 포함 엑셀 생성)
if len(rows) > 0:
    df = pd.DataFrame(rows)
else:
    df = pd.DataFrame(columns=[
        "filename","area_px","area_mm2","px_per_mm_x","px_per_mm_y",
        "conf","conf_mean","conf_max","n_instances","brightness_mean","blur_score",
        "bbox_w","bbox_h","perimeter_px","circularity","solidity","concavity","curvature","roughness"
    ])

df.to_excel(OUT_XLSX, index=False)
elapsed = time.time() - start_time
print(f"[완료] 결과 저장: {OUT_XLSX}")
print(f"총 처리 시간: {elapsed/60:.1f} 분")


[INFO] 총 grid cell 이미지 개수: 22536
[INFO] 총 8개의 chunk로 분할( chunk 당 3000장 )


Chunk 8/8: 100%|██████████| 64/64 [01:05<00:00,  1.02s/it]


[INFO] read_ok=22536  pred_batches=22536  with_masks=22536  total_instances=39546
[완료] 결과 저장: /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_윗면/4. 결과 출력 시각화/1작기(251128-251226)/1파트(251128-251213)/251212_lettuce_segmentation_area(1)_index.xlsx
총 처리 시간: 18.2 분
